# Lab 9: compare and cluster OpenAlex abstracts

In the lecture, we used vectors for words and documents. In this lab, you will work with the document side of that workflow.

## The question

Can a small embedding model find useful relationships among research abstracts? You will compare nearest documents, inspect a cluster, and check how much the answer changes when you change the model.

## What you will practice

By the end of the lab, you should be able to:

- explain what one row of a document-embedding table represents;
- use cosine similarity to find nearby abstracts;
- inspect cluster contents before naming a group;
- describe one result that changes when the model changes.

## Set up

Run these cells from top to bottom. If you restart the kernel, start here again.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
from sklearn.cluster import KMeans
from sklearn.decomposition import TruncatedSVD
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
DATA_BASE = Path("data")
if not DATA_BASE.exists():
    DATA_BASE = Path("..") / "data"

In [ ]:
if not DATA_BASE.exists():
    DATA_BASE = "https://raw.githubusercontent.com/macss-berkeley/compss-211a/main/data"

def data_path(filename):
    return DATA_BASE / filename if isinstance(DATA_BASE, Path) else f"{DATA_BASE}/{filename}"

## 1. Read the abstracts

This file contains 80 OpenAlex works with at least one UC Berkeley-affiliated authorship. That does not mean every work was led by Berkeley or written by Berkeley faculty.

In [ ]:
works = pd.read_csv(data_path("openalex_berkeley_abstracts_2024_sample.csv"))
works = works.dropna(subset=["abstract"]).reset_index(drop=True)
works.shape

In [ ]:
works[["title", "primary_domain", "primary_field"]].head()

In [ ]:
works["primary_domain"].value_counts(dropna=False)

Before modeling, note which domains appear most often. How might that imbalance affect the neighbors and clusters you find?

## 2. Build document vectors

TF-IDF makes a wide table of term weights. LSA compresses that table to eight columns. Each row still represents one abstract.

In [ ]:
vectorizer = TfidfVectorizer(stop_words="english", min_df=2, max_features=1500)
tfidf_matrix = vectorizer.fit_transform(works["abstract"])
tfidf_matrix.shape

In [ ]:
lsa = TruncatedSVD(n_components=8, random_state=211)
document_vectors = lsa.fit_transform(tfidf_matrix)
document_vectors.shape

In [ ]:
lsa.explained_variance_ratio_.sum()

The last number is the share of variation retained by the eight dimensions. Record it in your notes. What is gained by using eight columns, and what might be lost?

## 3. Find nearby abstracts

Choose one row as the anchor. Start with row 0, then change `anchor_id` to inspect a work that interests you.

In [ ]:
anchor_id = 1
works.loc[anchor_id, ["title", "primary_domain", "abstract"]]

In [ ]:
scores = cosine_similarity(document_vectors[anchor_id].reshape(1, -1), document_vectors)[0]
nearest_ids = scores.argsort()[::-1][1:4]

In [ ]:
neighbors = works.loc[nearest_ids, ["title", "primary_domain"]].copy()
neighbors["similarity"] = scores[nearest_ids]
neighbors

In [ ]:
works.loc[[anchor_id, nearest_ids[0]], ["title", "abstract"]]

### Your response

Read the anchor and its closest neighbor. What shared language or subject matter could explain the match? What makes the match questionable? Write two or three sentences.

## 4. Inspect a cluster

We will ask for five clusters. The numbers are labels assigned by the program, not rankings or known subject areas.

In [ ]:
model_5 = KMeans(n_clusters=5, n_init=20, random_state=211)
works["cluster_5"] = model_5.fit_predict(document_vectors)
works["cluster_5"].value_counts().sort_index()

In [ ]:
cluster_to_read = 0
works.loc[works["cluster_5"] == cluster_to_read, ["title", "primary_domain"]]

### Your response

Give the cluster a short, cautious description based on the titles. If the group is too mixed, say so and point to the titles that make it difficult to label.

## 5. Change the model

Run the same clustering with four groups. The cross-tab shows how documents move between the two solutions.

In [ ]:
model_4 = KMeans(n_clusters=4, n_init=20, random_state=211)
works["cluster_4"] = model_4.fit_predict(document_vectors)
pd.crosstab(works["cluster_4"], works["cluster_5"] )

### Your response

Describe one group that split, merged, or changed when you moved from five clusters to four. What does that change suggest about treating a cluster as a natural category?

## Exit check

Before you leave, make sure your notebook shows:

- the size of the TF-IDF and LSA tables;
- one anchor and its nearest documents;
- the contents of one cluster;
- your written responses about similarity and model choice.

Restart the kernel and run the whole notebook once. Fix any cell that depends on something you ran out of order.